# GreenTest: Python

This is the **Python** GreenTest. Every language in this repo follows the
same pattern - bootstrap the language, generate a small static site, serve
it locally, verify it's actually being served correctly - so bootstrapping
a new machine looks the same no matter which language you need next.

This one verifies [vanilla-compost](https://github.com/EcologyComputing/vanilla-compost),
the flagship, ground-floor demo app - the simplest possible static website.
If this is your first time in a terminal, writing Python, using git, or
looking at raw HTML, that's exactly who this notebook is for.

**New to any of this? Bookmark these for later, then come back:**
- Never used a terminal? [Software Carpentry: The Unix Shell](https://swcarpentry.github.io/shell-novice/)
- Never used git? [Pro Git, chapters 1-2](https://git-scm.com/book/en/v2)
- Never read Python before? [The Official Python Tutorial](https://docs.python.org/3/tutorial/)
- Curious what HTML actually is? [MDN: HTML basics](https://developer.mozilla.org/en-US/docs/Web/HTML)

Steps:
1. Point this notebook at your vanilla-compost clone
2. Confirm it's actually cloned (not just copied)
3. Leave a note about this run
4. Generate `posts.html` from the markdown posts
5. Serve the site locally
6. Verify the generated page is actually being served correctly
7. Clean up

Run the cells in order, top to bottom, using Shift+Enter. `generate_posts.py`
has no dependencies beyond the Python standard library, so there's nothing
to `pip install` for the test itself. (If you can't even open this notebook
yet, run `bootstrap.sh` in this same directory first.)

## 1. Point this notebook at your vanilla-compost clone

GreenTest assumes you cloned vanilla-compost as a sibling of this repo -
the same folder that has `greenTest/` also has `vanilla-compost/`. If
yours lives somewhere else, change the path below.

In [ ]:
import os

VANILLA_COMPOST = os.path.abspath("../../vanilla-compost")
os.environ['VANILLA_COMPOST'] = VANILLA_COMPOST
print(f"Testing vanilla-compost at: {VANILLA_COMPOST}")

## 2. Confirm the repo is cloned

The cell below runs in **bash** - the language of the terminal, not Python.
(In a notebook, a cell starting with `%%bash` switches to bash for that one
cell.) It checks that a couple of expected files exist, then checks that
the folder is a real `git clone` - a full copy of the project's history,
not just a folder of files someone emailed you. New to git?
[Pro Git, chapters 1-2](https://git-scm.com/book/en/v2) covers exactly this.

In [ ]:
%%bash
# A "git clone" is just a folder with a hidden .git directory tracking history.
if [ -f "$VANILLA_COMPOST/README.md" ] && [ -f "$VANILLA_COMPOST/src/generate_posts.py" ]; then
    echo "Repo layout looks right (found README.md and src/generate_posts.py)."
else
    echo "That path doesn't look like a vanilla-compost clone: $VANILLA_COMPOST"
    echo "Clone it first: git clone https://github.com/EcologyComputing/vanilla-compost.git"
    exit 1
fi

if git -C "$VANILLA_COMPOST" rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    echo "Confirmed: this is a real git clone, not just a folder of files."
    git -C "$VANILLA_COMPOST" remote get-url origin 2>/dev/null && echo "Its 'origin' remote points there ^ - that's where 'git pull' and 'git push' talk to." || echo "No 'origin' remote set - fine if you haven't pushed anywhere yet."
else
    echo "Warning: no .git found there. Fine for a quick trial, but you won't be able to track changes or send posts back via pull request without a real clone."
fi

## 3. Leave a note for this run

Edit the text between the `"""` marks below with anything about this run -
what you changed, what you're testing, why. That's just a Python string
(text between quote marks) - editing it doesn't require knowing any Python
beyond "this is text I can change." It gets appended to `greentest-log.md`
(with a timestamp) so there's a running record, not just a pass/fail.

In [ ]:
notes = """
Write your notes here before running the rest of the notebook.
"""

In [ ]:
from datetime import datetime

log_path = "greentest-log.md"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"## {timestamp}\n\n{notes.strip()}\n\n")

print(f"Notes appended to {log_path}")

## 4. Generate `posts.html`

This cell runs vanilla-compost's `generate_posts.py` - a small **Python**
program (the same language powering this notebook) that reads every file
in its `src/posts/` and writes `src/posts.html` listing them. New to
Python? The [official tutorial](https://docs.python.org/3/tutorial/) is a
solid, free starting point.

In [ ]:
import subprocess

result = subprocess.run(
    ["python3", os.path.join(VANILLA_COMPOST, "src", "generate_posts.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("generate_posts.py failed - see output above.")

Peek at what it generated - this is raw **HTML**, the markup language every
web page is built from. Curious what the tags mean?
[MDN's HTML docs](https://developer.mozilla.org/en-US/docs/Web/HTML) are the
standard reference.

In [ ]:
with open(os.path.join(VANILLA_COMPOST, "src", "posts.html"), encoding="utf-8") as f:
    generated = f.read()

# show just the generated post list, not the whole page
start = generated.find('<p class="lead">')
end = generated.find('</p>', start) + len('</p>')
print(generated[start:end] if start != -1 else generated)

## 5. Serve the site locally

This starts a tiny web server (`python -m http.server`) on your own
machine, so you can view the site in a browser exactly like a visitor
would - just at `http://localhost:8080/` instead of a real domain. It
runs in the background so the notebook can keep going; we'll stop it in
step 7.

In [ ]:
import time

server = subprocess.Popen(
    ["python3", "-m", "http.server", "8080"],
    cwd=os.path.join(VANILLA_COMPOST, "src"),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)  # give it a moment to start
print(f"Server started (pid {server.pid}) at http://localhost:8080/")

## 6. Verify it's serving the update correctly

This is the actual "green test": fetch the page from the server we just
started and check it matches what `generate_posts.py` wrote, and that the
sample post shows up. If both checks pass, you'll see a green message
below - proof the whole chain (write markdown -> generate HTML -> serve
it) works end to end on your machine.

In [ ]:
import urllib.request

with urllib.request.urlopen("http://localhost:8080/posts.html") as response:
    served = response.read().decode("utf-8")

assert served == generated, "Served posts.html doesn't match what generate_posts.py just wrote."
assert "hello-compost" in served, "Expected the hello-compost post to be listed."

print("Green: the server is serving the freshly generated posts.html.")

## 7. Clean up

In [ ]:
server.terminate()
server.wait()
print("Server stopped.")

If everything above ran without errors, Python is bootstrapped and working
end to end on this machine, verified against a real app instead of just
assumed. See `greentest-log.md` for a running history of these runs.

From here, `$VANILLA_COMPOST/README.md` picks up: edit its `src/posts/`
with your own content, push it to your own GitHub repo, and host it for
free on Netlify. See `../ECOLOGY.md` for how this fits into the rest of
the Ecology Computing methodology.